# Desafio de Analytics

Este notebook utiliza exclusivamente as tabelas da camada Gold para responder às perguntas de negócio propostas na atividade.

Os resultados são apresentados por meio de `display()`, utilizando a modelagem dimensional criada anteriormente.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## 1. Receita total dos filmes

**Pergunta:** Qual é a receita total, em reais, somada de todos os filmes da base?

Como a tabela fato possui um único registro por filme lançado, a receita total pode ser obtida diretamente pela soma de `receita_brl`.

In [0]:
display(spark.sql(f"""
    SELECT SUM(receita_brl) AS receita_lancados_brl,
           COUNT(*) AS filmes,
           COUNT(receita_brl) AS filmes_com_receita
    FROM {catalog}.gold.fact_movies_performance
"""))


receita_lancados_brl,filmes,filmes_com_receita
1000797210251.63,96463,3253


## 2. Cinco filmes com maior popularidade

**Pergunta:** Quais são os 5 filmes com maior popularidade?

A popularidade está armazenada na tabela fato, enquanto o título está na dimensão de filmes. As duas tabelas são relacionadas por `sk_movie_id`.

In [0]:
display(spark.sql(f"""
    SELECT m.titulo, f.popularidade
    FROM {catalog}.gold.fact_movies_performance f
    JOIN {catalog}.gold.dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC, m.titulo
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


## 3. Quantidade de filmes por gênero

**Pergunta:** Quantos filmes cada gênero possui?

Como filmes e gêneros possuem relacionamento muitos-para-muitos, a contagem utiliza `bridge_movie_genre`, evitando duplicações ou a necessidade de armazenar gêneros diretamente na tabela fato.

In [0]:
display(spark.sql(f"""
    SELECT g.nome_genero, COUNT(*) AS quantidade_filmes
    FROM {catalog}.gold.bridge_movie_genre b
    JOIN {catalog}.gold.dim_genres g ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY quantidade_filmes DESC, g.nome_genero
"""))

nome_genero,quantidade_filmes
Drama,30457
Documentary,18612
Comedy,17348
Thriller,9424
Horror,9163
Romance,6996
Action,5541
Crime,4320
Animation,4159
TV Movie,3676


## 4. Ranking dos 10 filmes de maior receita

**Pergunta:** Para os 10 filmes de maior receita, mostrar título, receita em US$ e R$ e a posição no ranking.

A função `RANK()` é aplicada sobre a receita em dólares em ordem decrescente. Dessa forma, filmes com a mesma receita recebem a mesma posição no ranking.

In [0]:
display(spark.sql(f"""
    SELECT
        titulo,
        receita_usd,
        receita_brl,
        posicao_ranking
    FROM (
        SELECT
            m.titulo,
            f.receita_usd,
            f.receita_brl,
            RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM {catalog}.gold.fact_movies_performance f
        JOIN {catalog}.gold.dim_movies m
            ON m.sk_movie_id = f.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    ORDER BY posicao_ranking, titulo
    LIMIT 10
"""))

titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,17336760000.00,1
Avatar: The Way of Water,2320250281.00,14366293664.87,2
AVENGERS: INFINITY WAR,2052415039.00,12707938196.98,3
spider-man: no way home,1921847111.00,11899500757.18,4
The Lion King,1663075401.00,10297263960.37,5
Top Gun: Maverick,1488732821.00,9217787007.79,6
Barbie,1428545028.00,8845122249.87,7
The Super Mario Bros. Movie,1355725263.00,8394244110.92,8
Black Panther,1349926083.00,8358337328.11,9
Star Wars: The Last Jedi,1332698830.00,8251671345.71,10


## Definição dos períodos de análise

Para as perguntas 5 e 6, a atividade determina que os períodos de 2 e 5 anos sejam calculados a partir da **data de lançamento válida mais recente da própria base**, e não da data atual.

Primeiramente são considerados somente filmes lançados, com data válida e não futura. A maior data restante é utilizada como referência para os dois recortes temporais.

In [0]:
df_filmes = spark.table(f"{catalog}.gold.dim_movies")
df_fato = spark.table(f"{catalog}.gold.fact_movies_performance")

# Considera somente filmes lançados com data válida e não futura
df_lancados = df_filmes.filter(
    (F.col("status_filme") == "Lançado") &
    F.col("data_lancamento").isNotNull() &
    (F.col("data_lancamento") <= F.current_date())
)

# Data mais recente válida presente na base
data_referencia = (
    df_lancados
    .agg(F.max("data_lancamento").alias("data_referencia"))
    .first()["data_referencia"]
)

print("Quantidade de filmes lançados válidos:", df_lancados.count())
print("Data de referência:", data_referencia)

# Filmes dos últimos 2 anos
df_ultimos_2_anos = (
    df_lancados
    .filter(
        F.col("data_lancamento") >=
        F.add_months(F.lit(data_referencia), -24)
    )
    .select("sk_movie_id")
)

# Filmes dos últimos 5 anos
df_ultimos_5_anos = (
    df_lancados
    .filter(
        F.col("data_lancamento") >=
        F.add_months(F.lit(data_referencia), -60)
    )
    .select("sk_movie_id")
)

Quantidade de filmes lançados válidos: 96461
Data de referência: 2026-02-19


## 5. Ator com maior número de participações nos últimos 2 anos

A tabela-ponte de pessoas é utilizada para relacionar filmes e atores.

Antes da contagem, o par filme–ator é deduplicado para garantir que cada participação seja contabilizada apenas uma vez. São consideradas somente entidades classificadas como `Ator`.

Caso exista empate na maior quantidade de participações, todos os atores empatados são apresentados.

In [0]:
# Participações dos atores
df_atores = (
    df_ultimos_2_anos
    .join(
        spark.table(f"{catalog}.gold.bridge_movie_person"),
        on="sk_movie_id",
        how="inner"
    )
    .join(
        spark.table(f"{catalog}.gold.dim_people"),
        on="sk_person_id",
        how="inner"
    )
    .filter(F.col("tipo_pessoa") == "Ator")
    .select("sk_movie_id", "sk_person_id", "nome_pessoa")
    .dropDuplicates(["sk_movie_id", "sk_person_id"])
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(
        F.count("*").alias("quantidade_filmes")
    )
)

# Maior quantidade de participações
df_max_ator = df_atores.agg(
    F.max("quantidade_filmes").alias("maior_quantidade")
)

# Mantém todos os atores empatados na primeira posição
df_atores_vencedores = (
    df_atores
    .join(
        df_max_ator,
        F.col("quantidade_filmes") == F.col("maior_quantidade"),
        how="inner"
    )
    .select(
        "nome_pessoa",
        "quantidade_filmes"
    )
    .orderBy("nome_pessoa")
)

display(df_atores_vencedores)

nome_pessoa,quantidade_filmes
Kevin Hart,64


## 6. Produtora com maior lucro nos últimos 5 anos

Os filmes do período são relacionados às produtoras através de `bridge_movie_company` e à tabela fato através de `sk_movie_id`.

O lucro em BRL é somado para todos os filmes associados a cada produtora. O par filme–produtora é deduplicado antes da agregação para evitar contabilizar o mesmo filme mais de uma vez.

Caso exista empate no maior lucro total, todas as produtoras empatadas são apresentadas.

In [0]:
# Calcula o lucro total de cada produtora no período
df_produtoras_periodo = (
    df_ultimos_5_anos
    .join(
        spark.table(f"{catalog}.gold.bridge_movie_company"),
        on="sk_movie_id",
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates(["sk_movie_id", "sk_company_id"])
    .join(
        spark.table(f"{catalog}.gold.dim_companies"),
        on="sk_company_id",
        how="inner"
    )
    .join(
        df_fato.select("sk_movie_id", "lucro_brl"),
        on="sk_movie_id",
        how="inner"
    )
    .groupBy("sk_company_id", "nome_produtora")
    .agg(
        F.sum("lucro_brl").alias("lucro_total_brl"),
        F.count("*").alias("filmes_no_periodo"),
        F.count("lucro_brl").alias("filmes_com_lucro_conhecido")
    )
    .filter(F.col("lucro_total_brl").isNotNull())
)

# Maior lucro
df_max_produtora = df_produtoras_periodo.agg(
    F.max("lucro_total_brl").alias("maior_lucro")
)

# Mantém todas as produtoras empatadas com o maior lucro
df_produtoras_vencedoras = (
    df_produtoras_periodo
    .join(
        df_max_produtora,
        F.col("lucro_total_brl") == F.col("maior_lucro"),
        how="inner"
    )
    .select(
        "nome_produtora",
        "lucro_total_brl",
        "filmes_no_periodo",
        "filmes_com_lucro_conhecido"
    )
    .orderBy("nome_produtora")
)

display(df_produtoras_vencedoras)

nome_produtora,lucro_total_brl,filmes_no_periodo,filmes_com_lucro_conhecido
Universal Pictures,35740533673.48,50,24
